### STAGE 1 — Build a Bias Index (continuous score)

We already have these columns (from headlines_annotated.csv):

- sentiment_score → ∈ [-1, 1]
- emotional_intensity → |sentiment_score|
- subjectivity → ∈ [0, 1]
- framing_score → integer ≥ 0
- loaded_language → yes / no
- country

This is more than enough.

### 🧠 Step 1: Normalize features (VERY important)
Some features are already bounded, others aren’t.
```bash
Feature	                                    Action
emotional_intensity                         already 0–1
subjectivity	                            already 0–1
framing_score	                            min–max normalize
loaded_language	                            yes→1, no→0
```

### ⚙️ Step 2: Bias Index Formula (final & defensible)

We define bias as a combination of emotion + opinion + framing:

```bash
Bias Index =
0.35 × subjectivity
+ 0.35 × emotional_intensity
+ 0.20 × framing_score_normalized
+ 0.10 × loaded_language_binary
```
Why this works?
```
Subjectivity + emotion = core bias
Framing = rhetorical bias
Loaded language = linguistic signal
Weights sum to 1 → interpretable
```

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("headlines_annotated.csv")
df.head()

,id,headline,country,label,headline_clean,sentiment_polarity,sentiment_score,emotional_intensity,subjectivity,framing_score,loaded_language
0,1,0% salary hike even after great performance? E...,India,Sensational/Clickbait,0 salary hike even after great performance emp...,positive,0.6249,0.6249,0.7500,1,yes
1,2,Learjet 45: The jet Ajit Pawar took for his fi...,India,Human-Interest,learjet 45 the jet ajit pawar took for his fin...,neutral,0.0000,0.0000,1.0000,0,no
2,3,Lead the next wave of consumer businesses with...,India,Promotional,lead the next wave of consumer businesses with...,neutral,0.0000,0.0000,0.0000,0,no
3,4,CUET UG 2026 registration ends in two days: Ho...,India,Neutral/Factual,cuet ug 2026 registration ends in two days how...,neutral,0.0000,0.0000,0.0000,0,no
4,5,Employee who was denied WFH by boss due to los...,India,Human-interest,employee who was denied wfh by boss due to los...,negative,-0.3818,0.3818,0.6875,1,yes


In [3]:
# binary encode loaded language
df["loaded_binary"] = df["loaded_language"].map({"yes": 1, "no": 0})

# normalize framing score (min-max)
fs_min = df["framing_score"].min()
fs_max = df["framing_score"].max()

df["framing_norm"] = (df["framing_score"] - fs_min) / (fs_max - fs_min + 1e-6)

df.head()

,id,headline,country,label,headline_clean,sentiment_polarity,sentiment_score,emotional_intensity,subjectivity,framing_score,loaded_language,loaded_binary,framing_norm
0,1,0% salary hike even after great performance? E...,India,Sensational/Clickbait,0 salary hike even after great performance emp...,positive,0.6249,0.6249,0.7500,1,yes,1,0.5
1,2,Learjet 45: The jet Ajit Pawar took for his fi...,India,Human-Interest,learjet 45 the jet ajit pawar took for his fin...,neutral,0.0000,0.0000,1.0000,0,no,0,0.0
2,3,Lead the next wave of consumer businesses with...,India,Promotional,lead the next wave of consumer businesses with...,neutral,0.0000,0.0000,0.0000,0,no,0,0.0
3,4,CUET UG 2026 registration ends in two days: Ho...,India,Neutral/Factual,cuet ug 2026 registration ends in two days how...,neutral,0.0000,0.0000,0.0000,0,no,0,0.0
4,5,Employee who was denied WFH by boss due to los...,India,Human-interest,employee who was denied wfh by boss due to los...,negative,-0.3818,0.3818,0.6875,1,yes,1,0.5


In [4]:
# bias index

df["bias_index"] = (
    0.35 * df["subjectivity"] +
    0.35 * df["emotional_intensity"] +
    0.20 * df["framing_norm"] +
    0.10 * df["loaded_binary"]
)

df["bias_index"].head()

0    0.681215
1    0.350000
2    0.000000
3    0.000000
4    0.574255
Name: bias_index, dtype: float64

In [8]:
df.to_csv("headlines_with_bias_index.csv", index=False)

print("✅ Bias Index computed successfully.")
print(df[["headline", "country", "bias_index"]].head())

✅ Bias Index computed successfully.
                                            headline country  bias_index
0  0% salary hike even after great performance? E...   India    0.681215
1  Learjet 45: The jet Ajit Pawar took for his fi...   India    0.350000
2  Lead the next wave of consumer businesses with...   India    0.000000
3  CUET UG 2026 registration ends in two days: Ho...   India    0.000000
4  Employee who was denied WFH by boss due to los...   India    0.574255
